## ***Getting ready and importing libraries***

In [2]:
import platform
import os

#Check the processor name
processor = platform.processor()
node_name = platform.node()

print(f"Computer Name: {node_name}")
print(f"Processor Info: {processor}")

#Check for a specific Google Colab environmental variable
if 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_BACKEND_VERSION' in os.environ:
    print("--- RESULT: You are on the GOOGLE COLAB CLOUD ---")
else:
    print("--- RESULT: You are on your LOCAL CHROMEBOOK ---")


Computer Name: ce6cd26035dc
Processor Info: x86_64
--- RESULT: You are on the GOOGLE COLAB CLOUD ---


In [3]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt

print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print("All libraries loaded successfully!")


NumPy: 2.0.2
Pandas: 2.2.2
Scikit-learn: 1.6.1
All libraries loaded successfully!


## **Data Exploration -- twinkering with it to find out what sort of data we dealing with**

In [4]:
#Read training data

trainingdata = pd.read_csv('titanic_train.csv') #create Panda DataFrame of training data

print(trainingdata.columns)

print(trainingdata.head)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')
<bound method NDFrame.head of      PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily

In [5]:
trainingdata = trainingdata.drop(['Name', 'Ticket', 'Cabin'], axis=1)

print(trainingdata.columns)

print(trainingdata.head)

Index(['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch',
       'Fare', 'Embarked'],
      dtype='object')
<bound method NDFrame.head of      PassengerId  Survived  Pclass     Sex   Age  SibSp  Parch     Fare  \
0              1         0       3    male  22.0      1      0   7.2500   
1              2         1       1  female  38.0      1      0  71.2833   
2              3         1       3  female  26.0      0      0   7.9250   
3              4         1       1  female  35.0      1      0  53.1000   
4              5         0       3    male  35.0      0      0   8.0500   
..           ...       ...     ...     ...   ...    ...    ...      ...   
886          887         0       2    male  27.0      0      0  13.0000   
887          888         1       1  female  19.0      0      0  30.0000   
888          889         0       3  female   NaN      1      2  23.4500   
889          890         1       1    male  26.0      0      0  30.0000   
890          891   

In [6]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

text_features = ['Sex', 'Embarked']


#spares_output=False makes sure a NumPy Array is returned instead of a sparse matrix
#handle_unknown='ignore' makes sure the program doesn't throw errors on new features in test data it never saw before

ct = ColumnTransformer( #Column Tranformer Pipeline for flexibility
    transformers=[
        ('drop_this', 'drop', ['PassengerId']), #temporarily hide "passengerid" from model without deleting from dataframe permanently
        ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), text_features),
    ],
    remainder='passthrough'
)

encodednumpyarray = ct.fit_transform(trainingdata)

new_cols = ct.get_feature_names_out()

trainingdata = pd.DataFrame(encodednumpyarray, columns=new_cols) #overwrite old training data with newly transformed training data


In [7]:
print(trainingdata.columns) #take note that the names of all columns have changed after pipeline column transformation

Index(['encoder__Sex_female', 'encoder__Sex_male', 'encoder__Embarked_C',
       'encoder__Embarked_Q', 'encoder__Embarked_S', 'encoder__Embarked_nan',
       'remainder__Survived', 'remainder__Pclass', 'remainder__Age',
       'remainder__SibSp', 'remainder__Parch', 'remainder__Fare'],
      dtype='object')


In [8]:
X_train = trainingdata.drop('remainder__Survived', axis=1) #X_train consist of all columns in training data except "remainder__Survived"

Y_train = trainingdata[['remainder__Survived']] #Y_train contains the labels (i.e. correct answers)


## **Model Tweaking and Exploration of Optimal Model Parameters**

In [10]:
#Train Random Forest on training dataset

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Define parameters for Random Forest
# n_estimators is the number of trees in the forest
parameters = {
    'n_estimators': [50, 100, 200],
    'max_depth': [4, 8, 10, 15, None],
    'min_samples_leaf': [1, 5, 10],
    'max_features': ['sqrt', 'log2', 0.33, 0.5, None] #sqrt means max num of features = sqrt(N).  log2 means max num of features = log2(N) where N is total num of features
}

# Initialize the Random Forest model
rf_model = RandomForestClassifier(random_state=42)

# Set up Grid Search
grid_search = GridSearchCV(rf_model, parameters, n_jobs=-1, cv=5, scoring='roc_auc')

# Fit the model
grid_search.fit(X_train, Y_train.values.ravel())

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Score: {grid_search.best_score_}")

Best Parameters: {'max_depth': 8, 'max_features': 0.5, 'min_samples_leaf': 5, 'n_estimators': 50}
Best Score: 0.8736174131327115


In [13]:
#Convert the results dictionary of diff combinations of decision tree parameters into a DataFrame
results_df = pd.DataFrame(grid_search.cv_results_)

#Sort the combinations from best (rank 1) to worst
results_df = results_df.sort_values(by='rank_test_score')

#Display top combinations to make it readable for me (AI Engineer) to see for transparency and in case i wanna tweak it
pd.set_option('display.max_colwidth', None) #Set max column width to None to show everything (prevent cutoff due to too long names in output)
display_columns = ['rank_test_score', 'mean_test_score', 'params']
print(results_df[display_columns].head(10)) # Shows the top 10


     rank_test_score  mean_test_score  \
75                 1         0.873617   
74                 2         0.872622   
76                 3         0.872495   
77                 4         0.871819   
85                 5         0.871656   
211                6         0.871650   
166                7         0.871517   
84                 8         0.871493   
167                9         0.871439   
72                10         0.871417   

                                                                                   params  
75       {'max_depth': 8, 'max_features': 0.5, 'min_samples_leaf': 5, 'n_estimators': 50}  
74      {'max_depth': 8, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 200}  
76      {'max_depth': 8, 'max_features': 0.5, 'min_samples_leaf': 5, 'n_estimators': 100}  
77      {'max_depth': 8, 'max_features': 0.5, 'min_samples_leaf': 5, 'n_estimators': 200}  
85     {'max_depth': 8, 'max_features': None, 'min_samples_leaf': 5, 'n_estimators': 100

In [14]:
#Read test data

testdata = pd.read_csv('titanic_test.csv') #create Panda DataFrame "data" of test data

print(testdata.columns)

print(testdata.head)

Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')
<bound method NDFrame.head of      PassengerId  Pclass                                          Name  \
0            892       3                              Kelly, Mr. James   
1            893       3              Wilkes, Mrs. James (Ellen Needs)   
2            894       2                     Myles, Mr. Thomas Francis   
3            895       3                              Wirz, Mr. Albert   
4            896       3  Hirvonen, Mrs. Alexander (Helga E Lindqvist)   
..           ...     ...                                           ...   
413         1305       3                            Spector, Mr. Woolf   
414         1306       1                  Oliva y Ocana, Dona. Fermina   
415         1307       3                  Saether, Mr. Simon Sivertsen   
416         1308       3                           Ware, Mr. Frederick   
417         130

In [15]:
testdata = testdata.drop(['Name', 'Ticket', 'Cabin'], axis=1)

print(testdata.columns)

print(testdata.head)

Index(['PassengerId', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked'],
      dtype='object')
<bound method NDFrame.head of      PassengerId  Pclass     Sex   Age  SibSp  Parch      Fare Embarked
0            892       3    male  34.5      0      0    7.8292        Q
1            893       3  female  47.0      1      0    7.0000        S
2            894       2    male  62.0      0      0    9.6875        Q
3            895       3    male  27.0      0      0    8.6625        S
4            896       3  female  22.0      1      1   12.2875        S
..           ...     ...     ...   ...    ...    ...       ...      ...
413         1305       3    male   NaN      0      0    8.0500        S
414         1306       1  female  39.0      0      0  108.9000        C
415         1307       3    male  38.5      0      0    7.2500        S
416         1308       3    male   NaN      0      0    8.0500        S
417         1309       3    male   NaN      1      1   22.3583  

In [16]:
passengerid = testdata[['PassengerId']] #for csv file submission later to kaggle's grader

#To prevent ValueError, we need to ensure the testdata has the same columns as the trainingdata that 'ct' was fitted on.
#Unfortunately, this includes a 'Survived' column which the test dataset does not have :)
#So be adaptive to come up with quick fix: We add a dummy 'Survived' column to testdata temporarily then remove it after we ColumnTransform!!

#IMPORTANT: Good habit to always put ".copy()" at the back cos for NumPy arrays and Pandas DataFrame,
#the "=" does reference assignment, NOT value assignment
#i.e. this means that if u modified testdata later on after this "=" assignment without ".copy()" behind,
#the testdata_for_transform's value would also follow suit and change cos "=" assigns reference, not value for almost all data structures in Python
#this is why the ".copy()" is SUPER IMPORTANT, you MUST always remember to put it!!!
testdata_for_transform = testdata.copy() #Work on a copy to avoid modifying original testdata unexpectedly
testdata_for_transform['Survived'] = 0 #Add a dummy placeholder 'Survived' column temporarily (note: remove this dummy 'Survived' column after we done)

encodednumpyarray = ct.transform(testdata_for_transform)

new_cols = ct.get_feature_names_out()

transformed_testdata_with_survived = pd.DataFrame(encodednumpyarray, columns=new_cols)

print(transformed_testdata_with_survived.columns)

#Drop the 'remainder__Survived' column to get the final X_test features
X_test = transformed_testdata_with_survived.drop('remainder__Survived', axis=1)

print(X_test.columns)


Index(['encoder__Sex_female', 'encoder__Sex_male', 'encoder__Embarked_C',
       'encoder__Embarked_Q', 'encoder__Embarked_S', 'encoder__Embarked_nan',
       'remainder__Survived', 'remainder__Pclass', 'remainder__Age',
       'remainder__SibSp', 'remainder__Parch', 'remainder__Fare'],
      dtype='object')
Index(['encoder__Sex_female', 'encoder__Sex_male', 'encoder__Embarked_C',
       'encoder__Embarked_Q', 'encoder__Embarked_S', 'encoder__Embarked_nan',
       'remainder__Pclass', 'remainder__Age', 'remainder__SibSp',
       'remainder__Parch', 'remainder__Fare'],
      dtype='object')


In [17]:
#grid_search is an object that contains all the experiments (all decision trees corresponding to all possible combinations of parameters)
#so we extract the best decision tree (using the .best_estimator_) as our final model
model = grid_search.best_estimator_

Y_predictions = model.predict(X_test) #Y_predictions is a NumPy Array that contains all the final answers

print(Y_predictions[:100]) #Output first 50 predictions to check that its working


[0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 0. 0. 1. 0. 1. 1. 0. 0. 0. 1. 0. 1. 1. 0.
 1. 0. 1. 0. 1. 0. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0.
 1. 1. 0. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 1. 1. 1. 0. 0. 1. 1. 0.
 0. 1. 1. 0. 0. 1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 1. 1. 0. 0. 1. 0. 0. 0.
 1. 0. 1. 0.]


## **Submit to Grader**

In [18]:
my_ans = pd.DataFrame(Y_predictions) #convert NumPy array to DataFrame

final_df_for_submission = pd.concat([passengerid, my_ans], axis=1) #Concat 2 dataframes

final_df_for_submission.columns = ["PassengerId", "Survived"] #Rename the column headers (from left to right)

final_df_for_submission['PassengerId'] = final_df_for_submission['PassengerId'].astype(int) #Convert the floats to integers (the Kaggle Grader is very inflexible and dictates must be integer)
final_df_for_submission['Survived'] = final_df_for_submission['Survived'].astype(int) #Convert the floats to integers (the Kaggle Grader is very inflexible and dictates must be integer)

final_df_for_submission.to_csv('final_results.csv', index=False) #index=False prevents an extra index column from being created


## **Kaggle Grader Score: 0.78229**